<a href="https://colab.research.google.com/github/HannahSHeil/ImageProcessingandAnalysis/blob/main/notebooks/CellSegmentation_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 From pixels to numbers: a guided nuclei and cell segmentation exercise

You have a raw two-channel fluorescence image of adherent cells: **H2B-mCherry** marks the nuclei, **GFP-tubulin** outlines the cell body. In this notebook you will turn that picture into a table: **how many cells, how big, how round, how bright.**

Adapted from © NEUBIAS (https://neubias.github.io/training-resources/workflow_nuclei_and_cells_segmentation/?id_activity_platform=imagej-macro-activity) by H.S.Heil, 04.10.2025. This version adds step-by-step guidance, prediction and check questions, and quality-control steps.

---

## 🎯 Learning objectives
After this exercise you can:
1. **Inspect** an image as data: shape, bit depth, intensity range, histogram, background level.
2. **Explain** what smoothing and thresholding do, and **predict** how their parameters change the result.
3. **Turn** a binary mask into labelled objects and **spot** typical errors (merged, missing, truncated objects).
4. **Explain** why a seeded watershed can separate touching cells when a threshold cannot.
5. **Extract** shape and intensity measurements, **interpret** them, and **recognise their traps** (background offset, border-touching objects, tiny objects).
6. **Document** your parameters so that someone else can reproduce your analysis.

## 🧭 How to work with this notebook
Run the cells **from top to bottom** (Shift + Enter). Do not skip the questions: they are the point of the exercise.
> 📌 **First, make your own copy:** *File → Save a copy in Drive*. Work only in your copy.

| Symbol | What it means |
|:--:|:--|
| 📖 | **Concept**: read before you run the next cell |
| 🔮 | **Predict**: write down your guess *before* you run the cell |
| 👀 | **Observe**: look carefully at the output and describe what you see |
| 🎛️ | **Explore**: play with the sliders |
| ✏️ | **Your turn**: a small coding task, with a built-in self-check |
| ✅ | **Check yourself**: click the arrow to reveal the answer (try first!) |
| 💡 | **Hint** |

> 💬 *To write down an answer, double-click on the text cell and type under **✍️ Your answer**.*

## 🗺️ The plan

```
 RAW IMAGE
    │
    ├─ 1. Look at the data          (what do the numbers look like?)
    │
    ├─ 2. NUCLEI channel            filter ─► threshold ─► label        ─► "seeds"
    │
    ├─ 3. CELL channel              filter ─► threshold ─► cell mask
    │                                         + seeds ─► watershed      ─► one label per cell
    │
    ├─ 4. Measure & interpret       shape, position, intensity ─► table ─► plots
    │
    └─ 5. Document & export         parameters + results
```

**Why this strategy?** Nuclei are compact and well separated, so they are easy to find, and every cell has one. The tubulin signal outlines the whole cell, but neighbouring cells touch, so a threshold alone cannot tell where one cell ends and the next begins. We therefore **find the nuclei first and use them as seeds to split the cell mask.**

## 0 · Setup
Run the cell below once. It installs/loads the libraries we need. *(You can ignore the output.)*

In [ ]:
#@title Step 0 – Install and import libraries { display-mode: "form" }
!pip install -q scikit-image tifffile ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import io, filters, measure, morphology, segmentation, color
import ipywidgets as widgets
from IPython.display import display

plt.rcParams["figure.dpi"] = 100
print("✅ Libraries loaded")

---
## 1 · Look at your data

📖 **Concept.** A digital image is not a picture, it is a **table of numbers** (an array). Each number is the signal the detector measured in one pixel. Colours and brightness on your screen are just a *display choice*. Before processing anything, always ask: *what do the numbers actually look like?*

### Step 1.1 – Load the image
We use a small, 2-channel crop of adherent cells. Run the cell and read the printed information.

In [ ]:
url = "https://github.com/NEUBIAS/training-resources/raw/master/image_data/xyc_16bit__nuclei_tubulin_crop.tif"
img = io.imread(url)

print("Array shape :", img.shape)
print("Data type   :", img.dtype)

# Split the stack into the two channels (the channel axis comes first)
cells  = img[0, ...]   # channel 1: GFP-tubulin  -> outlines the cell body
nuclei = img[1, ...]   # channel 2: H2B-mCherry  -> marks the nucleus

print()
for name, ch in [("cells", cells), ("nuclei", nuclei)]:
    print(f"{name:7s}: min = {ch.min()}, max = {ch.max()}, mean = {ch.mean():.0f}")

### 🔮👀 Question 1.1
1. What do the three numbers in the **shape** tell you? (Which one is the number of channels?)
2. The data type is `uint16`. What does that mean, and how many different intensity values can a pixel have?
3. Look at the **min** and **max** of each channel. Is anything surprising about them?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

1. The shape is **(channels, rows, columns)** = (2, 156, 163): two channels, each an image of 156 × 163 pixels.
2. `uint16` = *unsigned 16-bit integer*. Each pixel is a whole number between 0 and 65,535, so **65,536 possible grey levels** (an 8-bit image only has 256). Note that a pixel stores an *intensity*, not a colour.
3. The values do **not** start near 0: the minimum is around **33,000** in both channels, and the maximum is only ~1,200–1,300 higher. The useful signal occupies a narrow band of about 1,200 of the 65,536 available levels, sitting on a **constant offset**. (Offsets like this are common: detectors add a baseline so that noise never gives negative numbers.) Keep this in mind. It will matter again when we measure intensities!

Also note: the file contains **no pixel size**, so all lengths and areas in this notebook are in **pixels**, not µm.

</details>

### Step 1.2 – Look at both channels
👀 **Observe.** Run the cell. Note that the colour bars show the *real* intensity range of the data.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
titles = ["Channel 1 – Cells (GFP-tubulin)", "Channel 2 – Nuclei (H2B-mCherry)"]
for a, im, title in zip(ax, [cells, nuclei], titles):
    h = a.imshow(im, cmap="gray")
    a.set_title(title)
    a.axis("off")
    plt.colorbar(h, ax=a, fraction=0.046, label="intensity (a.u.)")
plt.tight_layout()
plt.show()

### 👀 Question 1.2
1. In the **nuclei** channel: roughly how many nuclei do you count? Are they all equally bright? Do all of them lie completely inside the image?
2. In the **cell** channel: do the cells touch each other?
3. Look at the position of the nuclei in the **tubulin** image. What do you see there, and why?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

1. About **12–13 nuclei**. They are **not equally bright**: some are bright, some clearly dimmer. Several are **cut by the image border** (left, bottom, right). Both facts will cause trouble later.
2. Yes, many cells **touch or overlap**, so there are no clean gaps between them.
3. Each nucleus shows up as a **dark hole** in the tubulin signal. Tubulin forms the cytoskeleton in the cytoplasm and is largely *excluded* from the nucleus.

This is also a good reminder to **always check which channel is which** before processing. If the channels were swapped, everything downstream would be wrong.

</details>

### Step 1.3 – Zoom in until you see the pixels
The cell below automatically zooms in on the **edge of a nucleus** and prints the pixel values.

In [ ]:
# Find a place where the intensity changes fastest (= a nucleus edge)
edge_strength = filters.sobel(nuclei.astype(float))
r, c = np.unravel_index(np.argmax(edge_strength), edge_strength.shape)
half = 4
r = int(np.clip(r, half, nuclei.shape[0] - half))
c = int(np.clip(c, half, nuclei.shape[1] - half))
patch = nuclei[r - half:r + half, c - half:c + half]

fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
ax[0].imshow(nuclei, cmap="gray")
ax[0].add_patch(plt.Rectangle((c - half - 0.5, r - half - 0.5), 2 * half, 2 * half,
                              fill=False, edgecolor="red", lw=2))
ax[0].set_title("Where we zoom in")
ax[0].axis("off")

ax[1].imshow(patch, cmap="gray", interpolation="nearest")
for (i, j), v in np.ndenumerate(patch):
    ax[1].text(j, i, str(v), ha="center", va="center", fontsize=8,
               color="white" if v < patch.mean() else "black")
ax[1].set_title("The same region as numbers (8 × 8 pixels)")
ax[1].axis("off")
plt.tight_layout()
plt.show()

### 👀 Question 1.3
Read the numbers from left (outside the nucleus) to right (inside), or the other way round. Does the intensity jump from "background" to "nucleus" **in one pixel**, or gradually? Where exactly is the "edge" of the nucleus?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

The intensity changes **gradually over a few pixels**. There is no single pixel where the nucleus "starts". This is the blur of the microscope (the **point spread function** from the earlier lectures) plus the pixel sampling.

Consequence: *the position of the edge, and therefore the measured size of the object, depends on which intensity level you call "edge"*. That is exactly what the **threshold** will decide. There is no "true" threshold, only a well-justified and consistently applied one.

</details>

### Step 1.4 – The histogram
A histogram counts how many pixels have each intensity. It is the main tool for choosing a threshold.

🔮 **Predict before you run:** The y-axis is logarithmic. Why do you think that is? And what do you expect the nuclei histogram to look like: one peak or two? Where would you place a threshold?

✍️ **Your guess:** *(double-click to write)*

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for a, im, title in zip(ax, [cells, nuclei], ["Cells (tubulin)", "Nuclei (H2B)"]):
    a.hist(im.ravel(), bins=200, color="gray")
    a.set_yscale("log")
    a.set_xlabel("pixel intensity")
    a.set_ylabel("number of pixels (log scale)")
    a.set_title(title)
plt.tight_layout()
plt.show()

<details>
<summary>✅ Check yourself – open after you have tried</summary>

* The **y-axis is logarithmic** because the numbers of pixels differ by orders of magnitude (the background has many more pixels than the nuclei). On a linear axis the interesting small populations would be invisible.
* The **nuclei** histogram has a big peak at low values (the **background**, around 33,000, not 0!) and a long tail towards higher values (the **nuclei**). There is no sharp second peak because the nuclei have *different brightness*. The tubulin histogram is more spread out because the cytoplasm varies in brightness.
* A threshold belongs somewhere **between the background peak and the nuclei**. Where exactly is what we will explore. Two different automatic methods will give different answers.

</details>

### Step 1.5 – Make working copies for segmentation
📖 **Concept: segment on a copy, measure on the original.**
We will rescale each channel to floating-point numbers between **0 and 1**. This makes parameters easy to understand ("threshold = 0.3" means 30 % of the way from the darkest to the brightest pixel) and keeps all the information in the narrow intensity band.

> ⚠️ **Important rule:** the processed copies (`cells_f`, `nuclei_f`) are used **only to find objects**. For **measurements** we will always go back to the **raw** arrays (`cells`, `nuclei`), because processing changes the intensity values.
>
> (If you squeezed this image into 8-bit instead, the ~1,200 useful levels would collapse into only about 10. Never throw away information before you analyse it!)

We also prepare **display versions** with stretched contrast. Changing the display never changes the data.

In [ ]:
def to_float01(a):
    """Rescale an array linearly so that its minimum -> 0 and its maximum -> 1."""
    a = a.astype(float)
    return (a - a.min()) / (a.max() - a.min())

# Working copies for segmentation (floats, 0..1)
cells_f  = to_float01(cells)
nuclei_f = to_float01(nuclei)

# Display versions: clip the brightest 0.5 % so that the picture looks good (display only!)
cells_disp  = np.clip(cells_f  / np.percentile(cells_f,  99.5), 0, 1)
nuclei_disp = np.clip(nuclei_f / np.percentile(nuclei_f, 99.5), 0, 1)

print("Distinct intensity values in the raw nuclei image:", len(np.unique(nuclei)))
print("Working copy nuclei_f: min", nuclei_f.min(), " max", nuclei_f.max())

### 🔧 Toolbox
The next cell bundles the steps you are about to learn into small **functions**, so the interactive sliders can reuse them. **Run it once. You don't need to read or change it** (but you may take a look at the end!).

In [ ]:
#@title 🔧 Toolbox – helper functions (run once, no need to change anything) { display-mode: "form" }

def remove_small(binary, min_size):
    """Remove connected objects smaller than `min_size` pixels from a binary mask."""
    lab = measure.label(binary, connectivity=1)
    sizes = np.bincount(lab.ravel())
    small = sizes < min_size
    small[0] = False                      # label 0 is the background, keep it
    return binary & ~small[lab]


def segment_nuclei(img_f, blur_radius=2, thresh=None, min_size=20):
    """Median filter -> threshold -> fill holes -> remove specks -> label.
    thresh=None (or 0) means: let Otsu's method choose the threshold."""
    if blur_radius > 0:
        smooth = ndi.median_filter(img_f, footprint=morphology.disk(blur_radius))
    else:
        smooth = img_f.copy()
    if not thresh:
        thresh = filters.threshold_otsu(smooth)
    binary = ndi.binary_fill_holes(smooth > thresh)
    binary = remove_small(binary, min_size)
    labels = measure.label(binary, connectivity=1)
    return smooth, binary, labels, thresh


def segment_cell_mask(img_f, blur_size=5, thresh=None, min_size=200):
    """Mean filter -> threshold -> fill holes -> remove specks. Returns a binary cell mask.
    thresh=None (or 0) means: let Li's method choose the threshold."""
    smooth = ndi.uniform_filter(img_f, size=blur_size)
    if not thresh:
        thresh = filters.threshold_li(smooth)
    binary = ndi.binary_fill_holes(smooth > thresh)
    binary = remove_small(binary, min_size)
    return smooth, binary, thresh


def outlines(base_gray, *layers):
    """Draw the outlines of one or more label images on a grey image.
    Each layer is a (labels, colour) pair, e.g. (nuclei_labels, (0, 1, 1))."""
    rgb = color.gray2rgb(base_gray)
    for labels, col in layers:
        rgb = segmentation.mark_boundaries(rgb, labels, color=col, mode="inner")
    return rgb


def compare_threshold_methods(image, title=""):
    """Show three automatic thresholds on the histogram and what each one finds."""
    methods = {"Otsu": filters.threshold_otsu,
               "Li": filters.threshold_li,
               "Triangle": filters.threshold_triangle}
    colours = {"Otsu": "tab:red", "Li": "tab:green", "Triangle": "tab:orange"}
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.hist(image.ravel(), bins=200, color="lightgray")
    ax.set_yscale("log")
    rows = []
    for name, func in methods.items():
        t = func(image)
        ax.axvline(t, color=colours[name], lw=2, label=f"{name}: {t:.3f}")
        rows.append({"Method": name,
                     "Threshold (0-1)": round(float(t), 3),
                     "Objects found": int(measure.label(image > t, connectivity=1).max()),
                     "Foreground (%)": round(100 * float((image > t).mean()), 1)})
    ax.set_xlabel("intensity (0-1)")
    ax.set_ylabel("number of pixels (log)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()
    display(pd.DataFrame(rows))

print("✅ Toolbox ready")

---
## 2 · Segment the nuclei

**Goal:** a **label image** in which every nucleus is one object with its own number. Four steps:

```
 raw nuclei ──► 1. filter (reduce noise) ──► 2. threshold (foreground / background)
            ──► 3. clean up (fill holes, drop tiny specks) ──► 4. label (one number per object)
```

### Step 2.1 – Filtering: why and how?
📖 **Concept.** Real images are noisy. A **filter** replaces each pixel by a value computed from its neighbourhood:

* **Mean filter**: the *average* of the neighbourhood. Smooth, but blurs edges, and one bright outlier pulls the average.
* **Median filter**: the *middle value* of the neighbourhood. Removes isolated outliers and keeps edges sharper.

The **radius** sets the size of the neighbourhood: larger radius means stronger smoothing, but also loss of small details.

🔮 **Predict.** Here is a tiny 3 × 3 "image" in which one pixel is a bright outlier (255), e.g. a hot pixel. If we replace the centre pixel by the **mean** or by the **median** of all 9 values, which result will be closer to the "true" background (~10)?

✍️ **Your guess:** *(double-click to write)*

In [ ]:
patch = np.array([[10,  12,  11],
                  [ 9, 255,  10],
                  [11,  10,  12]])

print("Mean   of the 9 pixels:", round(patch.mean(), 1))
print("Median of the 9 pixels:", np.median(patch))

<details>
<summary>✅ Check yourself – open after you have tried</summary>

The **median is 11**, very close to the background. The **mean is 37.8**, dragged up by the outlier. That is why the median filter is the standard choice for removing spikes and "salt-and-pepper" noise while keeping edges. We use it for the nuclei, whose edges we want to keep.

</details>

### Step 2.2 – The whole pipeline in slow motion
Run the cell. It shows each intermediate result with a **first guess** of parameters: median filter radius 2 and the threshold chosen automatically by **Otsu's method**.

> 📖 `measure.label(..., connectivity=1)` finds groups of **connected** foreground pixels (neighbours up/down/left/right, not diagonal) and gives each group its own integer: 1, 2, 3, ...

In [ ]:
demo_radius = 2
demo_smooth = ndi.median_filter(nuclei_f, footprint=morphology.disk(demo_radius))   # 1. filter
demo_thresh = filters.threshold_otsu(demo_smooth)                                    # 2. automatic threshold
demo_binary = demo_smooth > demo_thresh                                              #    apply it -> True / False
demo_labels = measure.label(demo_binary, connectivity=1)                             # 4. label objects

fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
ax[0].imshow(nuclei_disp, cmap="gray");                ax[0].set_title("Raw nuclei (display contrast)")
ax[1].imshow(demo_smooth, cmap="gray");                ax[1].set_title(f"1. Median filter (radius {demo_radius})")
ax[2].imshow(demo_binary, cmap="gray");                ax[2].set_title(f"2. Threshold = {demo_thresh:.3f} (Otsu) -> binary mask")
ax[3].imshow(color.label2rgb(demo_labels, bg_label=0)); ax[3].set_title(f"4. Labels: {demo_labels.max()} objects")
for a in ax: a.axis("off")
plt.tight_layout()
plt.show()

### 👀 Question 2.2
1. What is the difference between the **binary mask** (panel 3) and the **label image** (panel 4)? Why do we need the second one?
2. Compare the count in panel 4 with your own count from Question 1.2. Do they agree? If not, which nuclei look wrong, and in what way?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

1. In the **binary mask** every foreground pixel is simply "True", so all nuclei look the same. In the **label image** each connected object has its **own number** (shown as its own colour). Only then can we **count** objects and **measure each one separately**.
2. Otsu finds about **10 objects**, fewer than the ~12–13 nuclei you can see. Look at the **dim nucleus** in the middle left: it is badly broken up, and the dim nuclei at the right border are missing. We will find out why next.

</details>

### Step 2.3 – How does the computer choose a threshold?
Automatic methods read the **histogram** and pick a value following different rules:

* **Otsu**: separates the histogram into two classes so that the pixels within each class are as similar as possible. Works best when there are two clear populations of similar size.
* **Li**: finds the threshold that minimises the information lost ("cross-entropy") between the image and its binarised version.
* **Triangle**: draws a line from the peak of the histogram to its far end and takes the point of greatest distance. Good when one huge background peak dominates and the objects are a long, faint tail.

Run the cell to compare. The table shows how many objects each method finds.

In [ ]:
compare_threshold_methods(demo_smooth, title="Nuclei (median-filtered): three automatic thresholds")

### 🔮👀 Question 2.3
1. The three methods give **different thresholds and different object counts**. Which method gives the highest threshold?
2. We saw that the bright nuclei are nicely found but the dim ones are not. Using what you know about how Otsu works, why does a **single global threshold** struggle when objects have **different brightness**?
3. Is there one "correct" threshold? How would you decide?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

1. **Otsu** gives the highest threshold (about 0.26), then Li (~0.14), then Triangle (~0.07).
2. Otsu's rule is dominated by the **bright** objects. The threshold lands high up, between the background and the bright nuclei, and so falls *inside* the dimmer nuclei, which are then partly or completely lost. **A global threshold treats every pixel the same**, whatever the brightness of the object it belongs to.
3. No. You decide by **looking at the result** (Does each nucleus get one label? Does the outline sit on the visible edge?) and, ideally, by **comparing with a manual annotation**. Whatever you choose, apply the *same* rule to all images you want to compare. Next, you will explore this yourself.

</details>

### Step 2.4 – 🎛️ Explore: find good parameters yourself
Four sliders control the pipeline:

| Slider | What it does |
|--|--|
| **Noise** | *Adds artificial noise* to the image. This file is quite clean, so this lets you experience what the filter is for. Start at 0. |
| **Blur radius** | Radius of the median filter (0 = no filter). |
| **Threshold** | Intensity (0 to 1) above which a pixel counts as nucleus. It starts at the Otsu value. |
| **Min. size** | Objects smaller than this many pixels are discarded as specks. |

The four panels show: the filtered image, the histogram with your threshold, your outlines on the original, and the labelled result.

**Work through these experiments** (note what you see):

1. 🎛️ **Start with Noise 0, Blur 2, Threshold at the Otsu value.** Look at the dim nucleus in the middle left.
2. 🎛️ **Lower the threshold slowly** (towards ~0.10, then to ~0.03). What happens to the dim nuclei? And to the two bright nuclei at the bottom?
3. 🎛️ **Raise the threshold** towards 0.5 and beyond. What happens to the *size* of the outlines and to the *number* of objects?
4. 🎛️ **Set Noise to ~0.10**, Blur 0, Min. size 0. Then increase the blur radius, and then the min. size. What does each of them fix?
5. 🎛️ **Increase the blur radius to 8** (Noise 0). What happens to the outlines?
6. 🎛️ **Decide on your best combination** and write down the three values.

In [ ]:
# A fixed noise pattern, so that your results are reproducible
rng = np.random.default_rng(42)
noise_field = rng.normal(0, 1, nuclei_f.shape)

def explore_nuclei(noise, blur_radius, thresh, min_size):
    img_used = np.clip(nuclei_f + noise * noise_field, 0, 1)
    smooth, binary, labels, t = segment_nuclei(img_used, blur_radius, thresh, min_size)
    img_disp = np.clip(img_used / np.percentile(nuclei_f, 99.5), 0, 1)

    fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
    ax[0].imshow(smooth, cmap="gray", vmin=0, vmax=1)
    ax[0].set_title(f"Filtered (median, radius {blur_radius})")
    ax[1].hist(smooth.ravel(), bins=200, color="lightgray")
    ax[1].axvline(t, color="red", lw=2)
    ax[1].set_yscale("log")
    ax[1].set_title(f"Histogram, threshold = {t:.3f}")
    ax[2].imshow(outlines(img_disp, (labels, (1, 0.2, 0.2))))
    ax[2].set_title("Outlines on the (noisy) image")
    ax[3].imshow(color.label2rgb(labels, bg_label=0))
    ax[3].set_title(f"Labels: {labels.max()} objects")
    for i in (0, 2, 3): ax[i].axis("off")
    plt.tight_layout()
    plt.show()

t_otsu = float(filters.threshold_otsu(ndi.median_filter(nuclei_f, footprint=morphology.disk(2))))
sty = {"description_width": "initial"}
noise_sl  = widgets.FloatSlider(value=0,    min=0,    max=0.3, step=0.02,  description="Noise",       style=sty, continuous_update=False)
blur_sl   = widgets.IntSlider(  value=2,    min=0,    max=10,  step=1,     description="Blur radius", style=sty, continuous_update=False)
thresh_sl = widgets.FloatSlider(value=round(t_otsu, 3), min=0.01, max=1, step=0.005, readout_format=".3f",
                                description="Threshold", style=sty, continuous_update=False)
size_sl   = widgets.IntSlider(  value=20,   min=0,    max=300, step=5,     description="Min. size (px)", style=sty, continuous_update=False)

out = widgets.interactive_output(explore_nuclei, {"noise": noise_sl, "blur_radius": blur_sl,
                                                  "thresh": thresh_sl, "min_size": size_sl})
display(widgets.VBox([noise_sl, blur_sl, thresh_sl, size_sl]), out)

<details>
<summary>✅ What you should have seen – open after you experimented</summary>

1. At the Otsu value the dim nucleus in the middle left is reduced to a **fragment** and the dim nuclei at the right border are missing.
2. Lowering the threshold towards **~0.10–0.15** captures the dim nuclei with outlines that follow the visible edge. If you go too far (**~0.05 and below**), the **bright nuclei at the bottom merge** into one object through their blurred halos, and the outlines swell. At the extreme, the whole image becomes a single object.
3. Raising the threshold makes the nuclei **shrink** and, at some point, **disappear** one by one (the dim ones first). *The measured area depends directly on the threshold.*
4. With noise, a **blur radius of 0** leaves many tiny speckled objects. The **median filter** removes most of them, and the **min. size** removes any that remain. Filtering and size filtering fix *different* problems.
5. A large radius **shrinks and rounds** the outlines and makes small features disappear. This image has large nuclei, so the effect is moderate here, but it would be dramatic for small objects. Use only as much smoothing as you need.
6. A good combination for this image is around **blur radius 2, threshold 0.10–0.15, min. size 20**. Several values work; what matters is that the outlines sit on the visible edges and each nucleus gets exactly one label.

**Key lesson:** parameters are not "right" or "wrong". They are good or bad *for this image and this question*. Always look at your result.

</details>

### Step 2.5 – Fix your parameters and run the segmentation
Enter the values you chose. The defaults are a good starting point, so **change them if you found better ones**.

In [ ]:
#@markdown ### Nuclei segmentation parameters
blur_radius   = 2      #@param {type:"slider", min:0, max:10, step:1}
nuclei_thresh = 0.12   #@param {type:"number"}
min_size      = 20     #@param {type:"integer"}

In [ ]:
nuclei_smooth, nuclei_binary, nuclei_labels, nuclei_thresh_used = segment_nuclei(
    nuclei_f, blur_radius=blur_radius, thresh=nuclei_thresh, min_size=min_size)

# --- quick automatic quality check: objects much larger than the median may be merged nuclei
areas = np.bincount(nuclei_labels.ravel())[1:]
suspicious = np.where(areas > 1.8 * np.median(areas))[0] + 1
suspicious_mask = np.isin(nuclei_labels, suspicious)

fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
ax[0].imshow(outlines(nuclei_disp, (nuclei_labels, (1, 0.2, 0.2)), (suspicious_mask * nuclei_labels, (1, 1, 0))))
ax[0].set_title("Outlines (yellow = unusually large object)")
ax[1].imshow(color.label2rgb(nuclei_labels, bg_label=0))
ax[1].set_title(f"Final nuclei labels: {nuclei_labels.max()} objects")
for a in ax: a.axis("off")
plt.tight_layout()
plt.show()

print(f"Threshold used: {nuclei_thresh_used:.3f}")
print(f"Number of nuclei: {nuclei_labels.max()}")
print(f"Areas (pixels): smallest = {areas.min()}, median = {int(np.median(areas))}, largest = {areas.max()}")
print("Possible merged nuclei (label numbers):", suspicious.tolist() if len(suspicious) else "none")
print("\n✅ Checkpoint: you now have a label image called `nuclei_labels`.")

### 👀 Question 2.5 – Quality control
1. Does **each nucleus have exactly one label** (one colour)? Any merged or split nuclei?
2. Look at the printed **areas**. Is there a very small object? Where is it, and is it a whole nucleus?
3. Some nuclei **touch the image border**. Will their measured size be correct? Think ahead: how should we handle them when we measure?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

1. With reasonable parameters each nucleus gets one label. (If two nuclei were merged, the yellow warning would mark them, and the area would be about twice that of its neighbours. Splitting touching nuclei would need an additional step, such as a distance-transform watershed. See the challenge at the end if you are curious.)
2. Yes, there are a couple of **small objects at the image border**: these are only *slivers* of nuclei that continue outside the picture.
3. **No.** A nucleus cut by the border is only partly measured, so its area is too small, its shape is distorted, and its integrated intensity is too low. We will therefore **flag** border-touching objects in the measurement table and exclude them from shape and size statistics.

</details>

---
## 3 · Segment the cells

Nuclei done, now the cell bodies. The plan has two stages:

1. **Cell mask:** which pixels belong to *any* cell? (filter + threshold, just like for the nuclei)
2. **Split the mask into single cells:** use the nuclei as **seeds** and a **watershed** to draw the borders between neighbouring cells.

### Step 3.1 – Filter and threshold the tubulin channel
For the nuclei we used a *median* filter to keep sharp edges. The tubulin signal is textured and patchy, so for the cell mask we use a **mean filter** (also called a box or uniform filter) to even it out and get a smooth outline.

First, how do the automatic thresholds behave on this channel? Run the cell.

In [ ]:
cells_smooth_demo = ndi.uniform_filter(cells_f, size=5)
compare_threshold_methods(cells_smooth_demo, title="Cells (mean-filtered, size 5): three automatic thresholds")

### 🔮 Question 3.1
Which of the three thresholds do you expect to give the most sensible **cell mask**? Think about how much of the image is covered by cells (look at the picture again) and compare it with the **Foreground (%)** column. We will test your prediction in the explorer.

✍️ **Your guess:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

Looking at the image, cells cover roughly **half** of the field. **Otsu** (~36 % foreground) probably cuts too high and loses the dimmer cell edges, **Triangle** (~87 %) is far too low and covers almost everything. **Li** (~54 %) is the closest to what we can see. Check this yourself in the explorer by moving the threshold.

</details>

### Step 3.2 – 🎛️ Explore the cell mask
The sliders control the **mean filter size** (in pixels, use odd numbers), the **threshold** (it starts at Li's value), and the **minimum size**.

The panels show: the filtered image, the histogram, the mask outline (yellow) together with the **nuclei outlines** (cyan) on the tubulin image, and the **connected regions** of the mask (each region = one colour).

**Experiments:**
1. 🎛️ Start at Li's threshold. Does every **nucleus (cyan) lie inside the mask (yellow)**? Does the yellow outline follow the visible cell edges?
2. 🎛️ **Lower** the threshold step by step. What happens to the number of regions and to the yellow outline?
3. 🎛️ **Raise** it. What happens now? Do the cells stay in one piece?
4. 🎛️ Compare the **number of nuclei** with the **number of connected regions** that is printed below the picture.

In [ ]:
def explore_cells(blur_size, thresh, min_size):
    smooth, mask, t = segment_cell_mask(cells_f, blur_size, thresh, min_size)
    regions = measure.label(mask, connectivity=1)

    fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
    ax[0].imshow(smooth, cmap="gray", vmin=0, vmax=1)
    ax[0].set_title(f"Filtered (mean filter, size {blur_size})")
    ax[1].hist(smooth.ravel(), bins=200, color="lightgray")
    ax[1].axvline(t, color="red", lw=2)
    ax[1].set_yscale("log")
    ax[1].set_title(f"Histogram, threshold = {t:.3f}")
    ax[2].imshow(outlines(cells_disp, (mask.astype(int), (1, 1, 0)), (nuclei_labels, (0, 1, 1))))
    ax[2].set_title("Mask outline (yellow), nuclei (cyan)")
    ax[3].imshow(color.label2rgb(regions, bg_label=0))
    ax[3].set_title(f"Connected regions: {regions.max()}")
    for a in (ax[0], ax[2], ax[3]): a.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Nuclei found: {nuclei_labels.max()}   |   Connected regions in the cell mask: {regions.max()}")

t_li = float(filters.threshold_li(ndi.uniform_filter(cells_f, size=5)))
cblur_sl   = widgets.IntSlider(value=5, min=1, max=31, step=2, description="Filter size", style=sty, continuous_update=False)
cthresh_sl = widgets.FloatSlider(value=round(t_li, 3), min=0.02, max=0.9, step=0.005, readout_format=".3f",
                                 description="Threshold", style=sty, continuous_update=False)
csize_sl   = widgets.IntSlider(value=200, min=0, max=1000, step=50, description="Min. size (px)", style=sty, continuous_update=False)

out = widgets.interactive_output(explore_cells, {"blur_size": cblur_sl, "thresh": cthresh_sl, "min_size": csize_sl})
display(widgets.VBox([cblur_sl, cthresh_sl, csize_sl]), out)

<details>
<summary>✅ What you should have seen – open after you experimented</summary>

1. Around Li's value (~0.2) the yellow outline follows the visible cell borders and all the nuclei are inside it.
2. Lowering the threshold makes the mask **grow** into the dim surroundings, and neighbouring cells **fuse into bigger and bigger regions**, until the whole image is one single region (at ~0.1).
3. Raising it makes the mask **shrink**; dim cell parts (thin protrusions, flat edges) are lost and cells begin to **fall apart** into fragments, so the region count can even go *up*.
4. At a sensible threshold there are **fewer connected regions than nuclei**: touching cells have merged. This is the problem the watershed will solve. Just changing the threshold cannot fix it: if you raise it enough to separate the cells, you lose the dim parts of the cell.

</details>

### Step 3.3 – Fix the cell-mask parameters

In [ ]:
#@markdown ### Cell-mask parameters
cell_blur     = 5     #@param {type:"slider", min:1, max:31, step:2}
cell_thresh   = 0.2   #@param {type:"number"}
cell_min_size = 200   #@param {type:"integer"}

In [ ]:
cells_smooth, cells_binary, cell_thresh_used = segment_cell_mask(
    cells_f, blur_size=cell_blur, thresh=cell_thresh, min_size=cell_min_size)
cell_regions = measure.label(cells_binary, connectivity=1)

fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
ax[0].imshow(outlines(cells_disp, (cells_binary.astype(int), (1, 1, 0)), (nuclei_labels, (0, 1, 1))))
ax[0].set_title("Cell mask (yellow) and nuclei (cyan)")
ax[1].imshow(color.label2rgb(cell_regions, bg_label=0))
ax[1].set_title("Connected regions of the mask")
for a in ax: a.axis("off")
plt.tight_layout()
plt.show()

print(f"Nuclei                           : {nuclei_labels.max()}")
print(f"Connected regions in the mask    : {cell_regions.max()}")

### 👀 Question 3.3
1. Are there **more or fewer** connected regions than nuclei? Why?
2. Look at the right-hand picture: find regions that contain **more than one nucleus**. What would happen if we measured "cells" using these regions?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

1. **Fewer** regions than nuclei. A threshold only answers "is this pixel cell or not?". Where two cells touch, there is no gap, so they become **one connected region**.
2. Some regions contain two nuclei, so two cells. Measuring them would give **one object with double the area** and a wrong count. We need to **split** them, and the nuclei tell us how many cells there are and roughly where they are.

</details>

### Step 3.4 – Watershed: splitting touching cells
📖 **Concept.** Imagine the image as a **landscape** in which brightness is height. We **invert** the tubulin image, so that bright cell bodies become **valleys** and the dark gaps between cells become **ridges**.

Now place a **source of water at every seed** (one per nucleus) and let the water rise. Each seed floods its own valley. Where the water from two different seeds **meets**, we build a **dam**: that dam is the border between the two cells. We stop flooding outside the **cell mask**, so the background never gets flooded.

* **Seeds / markers** = nuclei labels → *how many cells and which ones*
* **Landscape** = inverted, smoothed tubulin image → *where the borders run*
* **Mask** = cell mask → *where cells may exist at all*

The cell below shows the idea in **1D**: an intensity profile through two touching cells. (Each cell has a dip: that's the nucleus, where tubulin is missing.)

In [ ]:
#@title 📖 Watershed in 1D { display-mode: "form" }
x = np.arange(240)
def bump(center, width, height=1.0): return height * np.exp(-(x - center) ** 2 / (2 * width ** 2))
profile = bump(70, 28) + 0.9 * bump(170, 28)          # two touching cells
profile = profile - 0.45 * bump(70, 9) - 0.40 * bump(170, 9)   # dim nuclei in the middle of each cell
profile = profile + 0.03

landscape = -profile                                    # invert: bright -> low
markers = np.zeros_like(x); markers[70] = 1; markers[170] = 2   # one seed per nucleus
ws = segmentation.watershed(landscape, markers)
border = int(np.where(np.diff(ws) != 0)[0][0])

fig, ax = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
ax[0].plot(x, profile, "k")
ax[0].fill_between(x, profile, where=ws == 1, color="tab:blue", alpha=0.4, label="cell 1")
ax[0].fill_between(x, profile, where=ws == 2, color="tab:orange", alpha=0.4, label="cell 2")
ax[0].plot([70, 170], [profile[70], profile[170]], "v", color="red", ms=10, label="seeds (nuclei)")
ax[0].set_ylabel("tubulin intensity"); ax[0].legend(loc="upper right")
ax[0].set_title("Intensity profile across two touching cells")

ax[1].plot(x, landscape, "k")
ax[1].plot([70, 170], [landscape[70], landscape[170]], "v", color="red", ms=10)
ax[1].axvline(border, color="green", lw=3, ls="--")
ax[1].text(border + 4, landscape.max() - 0.15, "dam = cell border", color="green", fontsize=11)
ax[1].set_ylabel("inverted = landscape"); ax[1].set_xlabel("position along the line (pixels)")
ax[1].set_title("Inverted: cells are valleys, the dark gap between them is a ridge")
plt.tight_layout()
plt.show()

Notice that the border ends up on the **darkest point between the two cells** (the ridge), exactly where we would draw it by hand. The seeds don't need to be at the bottom of a valley, because the flooding always continues in the order of the landscape height.

### Step 3.5 – Run the watershed
Now in 2D. In skimage the three ingredients are the function arguments: the **landscape** (first), `markers=` and `mask=`.

In [ ]:
cell_landscape = -cells_smooth          # invert: bright cell body -> low valley

cell_labels = segmentation.watershed(
    cell_landscape,
    markers=nuclei_labels,              # one seed per nucleus
    mask=cells_binary)                  # only flood inside the cell mask

fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
ax[0].imshow(color.label2rgb(cell_regions, image=cells_disp, alpha=0.45, bg_label=0))
ax[0].set_title(f"BEFORE: connected regions ({cell_regions.max()})")
ax[1].imshow(color.label2rgb(cell_labels, image=cells_disp, alpha=0.45, bg_label=0))
ax[1].set_title(f"AFTER: watershed ({len(np.unique(cell_labels)) - 1} cells)")
for a in ax: a.axis("off")
plt.tight_layout()
plt.show()

### 🔮 Question 3.5
Before you open the answers, try to explain in your own words:
1. Why do we **invert** the image (`-cells_smooth`)?
2. Why do we pass the **nuclei labels as markers**, and what is the benefit of the numbers carrying over?
3. What would happen **without** `mask=`?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

1. Watershed floods from **low to high**. We want the bright cell bodies to be the low points (valleys) and the dark gaps to be the high points (ridges), so the borders end up on the gaps.
2. One seed per nucleus means **exactly one region per cell**, so no over-splitting. Because the labels are carried over, **cell number *i* belongs to nucleus number *i***. That lets us combine nucleus and cell measurements per cell later (e.g. the nucleus-to-cell area ratio).
3. Without a mask the flooding fills the **entire image**, background included, so every cell would be assigned a huge area up to the image edge.

</details>

### Step 3.6 – Check your segmentation (automatic quality control)
Watershed is only as good as its seeds and its mask, so we ask a few questions:
* Did **every nucleus** end up with a cell? (A nucleus outside the cell mask gets no cell.)
* How much of the cell mask is **not claimed** by any nucleus? (e.g. cells whose nucleus lies outside the cropped image)

In [ ]:
nuc_ids  = set(np.unique(nuclei_labels)) - {0}
cell_ids = set(np.unique(cell_labels)) - {0}
nuclei_without_cell = sorted(int(i) for i in nuc_ids - cell_ids)
unclaimed = cells_binary & (cell_labels == 0)

print(f"Nuclei: {len(nuc_ids)}   |   Cells after watershed: {len(cell_ids)}")
print("Nuclei without a cell:", nuclei_without_cell if nuclei_without_cell else "none ✅")
print(f"Cell-mask area not claimed by any nucleus: {unclaimed.sum()} px ({100 * unclaimed.sum() / cells_binary.sum():.1f} % of the mask)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(outlines(cells_disp, (cell_labels, (1, 1, 0)), (unclaimed.astype(int), (1, 0, 1))))
ax.set_title("Unclaimed cell area (magenta)")
ax.axis("off")
plt.show()

### Step 3.7 – Overlay the final result on the raw data
The most important quality-control step is simply **looking at the result on top of the original image.** Yellow = cell borders, cyan = nuclei.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(cells_disp, cmap="gray");  ax[0].set_title("Raw tubulin")
ax[1].imshow(outlines(cells_disp, (cell_labels, (1, 1, 0)), (nuclei_labels, (0, 1, 1))))
ax[1].set_title(f"Final segmentation: {len(np.unique(cell_labels)) - 1} cells")
for a in ax: a.axis("off")
plt.tight_layout()
plt.show()
print("✅ Checkpoint: you now have `nuclei_labels` and `cell_labels` (same numbers = same cell).")

### 👀 Question 3.7 – Be your own reviewer
Find **at least two places** where the segmentation is not perfect, and describe each as precisely as you can. Use these terms:
**merged** (two objects as one) · **split** (one object as two) · **missing** (object not detected) · **border wrong** (outline too big / too small / misplaced) · **truncated by the image border**.

Which of your findings would matter for your measurements? Which could you fix by changing a parameter, and which not?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

Typical findings for this image:
* **Cells without a nucleus in the crop**: a bright elongated cell at the right and a few cells at the border are in the mask but have **no label** (the magenta area above). They are simply not measured. That is expected for a crop, but you should know about it.
* **Cell borders** between neighbours follow the dark line between cells well, but in places a border is **slightly misplaced** where the dark line is faint.
* **Cells and nuclei truncated by the image border** have incomplete areas.
* The **bright ring-shaped cell** (round, in the middle-right) is a rounded cell with tubulin concentrated at its rim, which shows how the tubulin pattern differs between cells.
* Some nuclei **carry small attached extras** (bright buds), which are included in their outline.

**No segmentation is perfect.** What matters is that you (1) **look** at it, (2) know **what kind of errors** it makes, (3) judge whether they **bias** your measurement, and (4) **report** them honestly. For many errors, tuning the parameters helps. For others (cells without a visible nucleus, cut-off cells) you need to **exclude** objects or use different data.

</details>

---
## 4 · Measure and interpret

Segmentation is not the goal, it is the **means**. Now that every nucleus and every cell has its own label, we can measure each one.

### Step 4.1 – Measure the nuclei
📖 **Concept.** `measure.regionprops` goes through the label image and computes properties of each object. Here is what we extract:

| Measurement | Meaning | Where it comes from |
|--|--|--|
| **Area** | number of pixels in the object | label image |
| **Circularity** | $4\pi \cdot \text{Area} / \text{Perimeter}^2$ (1 = perfect circle, lower = more irregular or elongated) | label image |
| **Major / Minor axis length** | long and short axis of the ellipse that has the same second moments as the object | label image |
| **Centroid X / Y** | centre of mass (position) of the object | label image |
| **H2B Signal (raw)** | sum of the raw H2B intensities of all pixels of the object | label image **+ raw intensity image** |
| **Touches border** | does the object touch the image edge? (then it is cut off!) | label image |

> 📏 **Units.** This file has no pixel-size information, so lengths are in **pixels** and areas in **pixels²**. Here we will assume 100 nm/px.

In [ ]:
#@markdown Pixel size in µm (leave 0 if unknown):
pixel_size_um = 0.1   #@param {type:"number"}

In [ ]:
H, W = nuclei.shape
rows = []
for r in measure.regionprops(nuclei_labels, intensity_image=nuclei):   # <- RAW image for intensities
    min_r, min_c, max_r, max_c = r.bbox
    rows.append({
        "Label": r.label,
        "Area": int(r.area),
        "Circularity": 4 * np.pi * r.area / r.perimeter ** 2 if r.perimeter > 0 else np.nan,
        "Major Axis Length": r.major_axis_length,
        "Minor Axis Length": r.minor_axis_length,
        "Centroid X": r.centroid[1],          # centroid is (row, column) = (y, x)
        "Centroid Y": r.centroid[0],
        "H2B Signal (raw)": float(nuclei[tuple(r.coords.T)].sum()),   # add up the raw intensities of the object's pixels
        "Touches border": bool(min_r == 0 or min_c == 0 or max_r == H or max_c == W),
    })
df = pd.DataFrame(rows)
if pixel_size_um > 0:
    df["Area (µm²)"] = df["Area"] * pixel_size_um ** 2

summary_cols = ["Area", "Circularity", "Major Axis Length", "Minor Axis Length", "H2B Signal (raw)"]
print(f"Total number of segmented nuclei: {len(df)}  ({df['Touches border'].sum()} touch the image border)\n")
print("Mean and standard deviation (all nuclei):")
display(df[summary_cols].agg(["mean", "std"]).round(2))
print("The full table:")
display(df.round(2))

### Step 4.2 – ⚠️ A trap: is the intensity measurement meaningful?
H2B is a histone: it binds DNA. A nucleus with more DNA should contain more H2B, but **a bigger nucleus also simply covers more pixels**. So what do we expect when we plot the **H2B Signal (raw)** against the **Area**?

🔮 **Predict.** Will the points scatter widely, follow a trend with some spread, or lie almost perfectly on a line? Remember what you learned about the intensity values in Step 1.1.

✍️ **Your guess:** *(double-click to write)*

In [ ]:
corr = df["Area"].corr(df["H2B Signal (raw)"])

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(df["Area"], df["H2B Signal (raw)"], c="tab:blue", edgecolors="k")
ax.set_xlabel("Area (pixels)"); ax.set_ylabel("H2B Signal (raw)")
ax.set_title(f"Correlation coefficient = {corr:.4f}")
ax.grid(True)
plt.show()

### 👀 Question 4.2
1. What do you notice? Is this what you would expect from a measurement of *how much H2B* a nucleus contains?
2. Hint: remember Step 1.1. What was the **background level** in the nuclei image? What does the sum of the raw intensities contain, besides the H2B signal?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

1. The points lie **on an almost perfect straight line** (correlation ≈ 1.000). That would only be the case if every nucleus had exactly the same brightness per pixel. It is a sign that the measurement is **not measuring H2B**.
2. The background is not 0 but about **33,000**. Summing the raw values therefore adds ~33,000 for *every pixel* of the nucleus, and this baseline is ~**99 %** of the total. The "signal" is basically *Area × 33,000*. It is the area again, not the H2B content.

**Lesson:** *always subtract the background before measuring intensities*. Otherwise the offset dominates and the measurement just reflects object size.

</details>

### Step 4.3 – Subtract the background and measure again
We estimate the background level from the pixels **far away from any nucleus** (here: at least 8 pixels from a segmented nucleus), take their **median**, and subtract it from each pixel's raw value.

In [ ]:
# Background = median raw intensity of the pixels that are at least 8 px away from any nucleus
far_from_nuclei = ~ndi.binary_dilation(nuclei_labels > 0, structure=morphology.disk(8))
background = np.median(nuclei[far_from_nuclei])
print(f"Estimated background: {background:.0f}   (from {far_from_nuclei.sum()} pixels)")

df["H2B Signal (bg-corrected)"] = df["H2B Signal (raw)"] - background * df["Area"]
df["H2B Mean (bg-corrected)"]   = df["H2B Signal (bg-corrected)"] / df["Area"]

print(f"Share of the raw signal that was just background: {100 * (background * df['Area'] / df['H2B Signal (raw)']).mean():.1f} %")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for a, col in zip(ax, ["H2B Signal (raw)", "H2B Signal (bg-corrected)"]):
    a.scatter(df["Area"], df[col], c="tab:blue", edgecolors="k")
    a.set_xlabel("Area (pixels)"); a.set_ylabel(col)
    a.set_title(f"{col}\ncorrelation with area = {df['Area'].corr(df[col]):.3f}")
    a.grid(True)
plt.tight_layout()
plt.show()

<details>
<summary>✅ Check yourself – open after you have tried</summary>

After the correction the points are **no longer on a perfect line**: bigger nuclei tend to contain more H2B in total (as expected, they have more chromatin), but there is real **spread**, the biological and technical variation we want to measure. The nuclei that are **cut by the image border** have too little signal because part of them is missing, which is another reason to treat border objects separately.

</details>

### Step 4.4 – 🎛️ Explore the data
Pick any two measurements and plot them against each other. Border-touching nuclei are shown in **orange**.

**Try these three:**
1. **Area** vs. **H2B Signal (bg-corrected)**: what is the relationship?
2. **Major Axis Length** vs. **Minor Axis Length**: what does the position relative to the diagonal tell you?
3. **Area** vs. **Circularity**: where are the small objects, and how reliable is their circularity?

In [ ]:
features = ["Area", "Circularity", "Major Axis Length", "Minor Axis Length",
            "Centroid X", "Centroid Y", "H2B Signal (raw)", "H2B Signal (bg-corrected)", "H2B Mean (bg-corrected)"]

def plot_scatter(x_feature, y_feature):
    edge = df["Touches border"]
    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.scatter(df.loc[~edge, x_feature], df.loc[~edge, y_feature], c="tab:blue", edgecolors="k", alpha=0.8, label="inside image")
    ax.scatter(df.loc[edge,  x_feature], df.loc[edge,  y_feature], c="tab:orange", edgecolors="k", alpha=0.8, label="touches border")
    ax.set_xlabel(x_feature); ax.set_ylabel(y_feature)
    ax.set_title(f"{x_feature} vs. {y_feature}")
    ax.grid(True); ax.legend()
    plt.show()

x_dd = widgets.Dropdown(options=features, value="Area", description="X feature:", style=sty)
y_dd = widgets.Dropdown(options=features, value="H2B Signal (bg-corrected)", description="Y feature:", style=sty)
out = widgets.interactive_output(plot_scatter, {"x_feature": x_dd, "y_feature": y_dd})
display(widgets.HBox([x_dd, y_dd]), out)

<details>
<summary>✅ What you should have seen – open after you explored</summary>

1. **Positive relationship with spread**: larger nuclei hold more H2B in total. H2B amount is roughly proportional to the amount of DNA, which is why integrated H2B intensity is often used as a rough proxy for DNA content (and thus for cell-cycle stage). Border-touching nuclei fall below the trend because part of the signal is cut off.
2. The major axis is always ≥ the minor axis, so **all points lie on or above the diagonal**. Points **close to the diagonal** are round nuclei, points **far above** are elongated.
3. Small objects (mostly border slivers) have few pixels, so their **perimeter is poorly defined** and the circularity is **unreliable**, which brings us to the next question.

</details>

### Step 4.5 – Circularity: what does it tell us, and where does it fail?
$$\text{Circularity} = \frac{4\pi \cdot \text{Area}}{\text{Perimeter}^2}$$

### 🔮 Question 4.5
1. What does a circularity **close to 1** tell you about the shape of a nucleus?
2. What does a value **much smaller than 1** tell you?
3. A circle gives exactly 1, and no shape should give *more* than 1. Yet you may see values **greater than 1**. Why could that happen?

> 💡 **Hint:** think about how the **perimeter** of an object made of square pixels is estimated, especially for **very small** objects and for objects at the **image border**.

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

1. A shape close to a **circle** (round nucleus).
2. An **elongated or irregular** shape (a long ellipse, a lobed or dented outline, or an object with protrusions).
3. The perimeter of a pixelated object is only an **estimate**: a staircase of pixels is not a smooth curve. For **small objects** the estimate can be too short compared with the area, so the ratio can exceed 1 slightly. Cut-off objects at the image border have unnatural, truncated shapes too. A circularity above 1 is therefore a **warning sign for a tiny or truncated object**, not a real property of the nucleus.

</details>

### Step 4.6 – ✏️ Your turn (1): find the nucleus with circularity > 1
Let's find out which nucleus has a circularity larger than 1. We use a pandas **table filter**:
`df[ condition ]` returns only the rows for which the condition is `True`, e.g. `df[ df["Area"] > 500 ]` gives all nuclei larger than 500 pixels.

Complete the line in the next cell. Replace `None` by your filter.

> 💡 **Hint:** the column is called `"Circularity"` and you want it to be greater than 1.

<details>
<summary>💡 Solution – only after you tried</summary>

```python
high_circ = df[ df["Circularity"] > 1 ]
```

</details>

In [ ]:
# ✏️ Select the rows of the table where the circularity is larger than 1
high_circ = None      # <-- replace None with your filter:  df[ ... ]

# ---- self-check (do not change) ----
if high_circ is None:
    print("✋ Complete the line above (replace None), then run this cell again.")
else:
    print(f"{len(high_circ)} nucleus/nuclei with circularity > 1:")
    display(high_circ[["Label", "Area", "Circularity", "Touches border"]].round(3))

Now **look at** the nucleus you found. Run the next cell. It outlines it in yellow and zooms in.

### 👀 Question 4.6
What do you see? What do the object(s) with circularity > 1 have in common (size, position)? Would you keep them in your analysis?

✍️ **Your answer:** *(double-click to write)*

In [ ]:
if high_circ is None or len(high_circ) == 0:
    print("Nothing to show yet. Complete the previous cell first. "
          "(If the table was empty, no nucleus has circularity > 1 with your parameters.)")
else:
    highlight = np.isin(nuclei_labels, high_circ["Label"].values)
    r0, c0, r1, c1 = measure.regionprops(highlight.astype(int))[0].bbox
    pad = 12
    sl = (slice(max(r0 - pad, 0), r1 + pad), slice(max(c0 - pad, 0), c1 + pad))

    both = outlines(nuclei_disp, (nuclei_labels, (1, 0.2, 0.2)), (highlight * nuclei_labels, (1, 1, 0)))
    fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
    ax[0].imshow(both);                          ax[0].set_title("Highlighted in yellow")
    ax[1].imshow(both[sl], interpolation="nearest"); ax[1].set_title("Zoom (you can see the pixels)")
    for a in ax: a.axis("off")
    plt.tight_layout()
    plt.show()

<details>
<summary>✅ Check yourself – open after you have tried</summary>

The object is **small** and **touches the image border**: it is only a sliver of a nucleus that lies mostly outside the picture. Its perimeter is estimated from just a handful of pixels, and the shape is a truncated one. The circularity > 1 is a numerical artefact, not biology. In an analysis of shape or size you would **exclude border-touching objects** (use the "Touches border" column) and tiny ones.

</details>

### Step 4.7 – ✏️ Your turn (2): measure the cells and compute the N/C ratio
Because the watershed carried the nucleus numbers over, **cell number *i* belongs to nucleus number *i*** and we can combine the two. A classic cell-biology readout is the **nuclear-to-cytoplasmic (N/C) area ratio**: the area of the nucleus divided by the area of its cell.

The next cell measures the cells for you. Your job: write the formula for the ratio.

> 💡 **Hint:** `df["Area"]` is the nuclear area, `df["Cell Area"]` is the area of the cell. A column divided by another column is calculated row by row.

<details>
<summary>💡 Solution – only after you tried</summary>

```python
nc_ratio = df["Area"] / df["Cell Area"]
```

</details>

In [ ]:
# --- measure the cells (given) ---
cell_info = {}
for r in measure.regionprops(cell_labels):
    min_r, min_c, max_r, max_c = r.bbox
    cell_info[r.label] = (int(r.area), bool(min_r == 0 or min_c == 0 or max_r == H or max_c == W))
df["Cell Area"]           = df["Label"].map(lambda i: cell_info.get(i, (np.nan, False))[0])
df["Cell touches border"] = df["Label"].map(lambda i: cell_info.get(i, (np.nan, False))[1])

# ✏️ Nuclear-to-cell area ratio
nc_ratio = None       # <-- replace None with a formula using df["Area"] and df["Cell Area"]

# ---- self-check (do not change) ----
if nc_ratio is None:
    print("✋ Complete the line above (replace None), then run this cell again.")
else:
    df["N/C Area Ratio"] = nc_ratio
    complete = ~(df["Touches border"] | df["Cell touches border"])
    print(f"{complete.sum()} of {len(df)} cells are complete (neither nucleus nor cell touches the border).")
    print(f"N/C ratio of the complete cells: mean = {df.loc[complete, 'N/C Area Ratio'].mean():.2f}, "
          f"std = {df.loc[complete, 'N/C Area Ratio'].std():.2f}\n")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].scatter(df.loc[complete, "Cell Area"],  df.loc[complete, "Area"],  c="tab:blue",   edgecolors="k", label="complete")
    ax[0].scatter(df.loc[~complete, "Cell Area"], df.loc[~complete, "Area"], c="tab:orange", edgecolors="k", label="cut by border")
    ax[0].set_xlabel("Cell area (pixels)"); ax[0].set_ylabel("Nucleus area (pixels)")
    ax[0].set_title("Nucleus area vs. cell area"); ax[0].grid(True); ax[0].legend()
    ax[1].hist(df.loc[complete, "N/C Area Ratio"], bins=8, color="tab:blue", edgecolor="k")
    ax[1].set_xlabel("N/C area ratio"); ax[1].set_ylabel("number of cells")
    ax[1].set_title("N/C ratio (complete cells only)")
    plt.tight_layout()
    plt.show()

### 👀 Question 4.7
1. Why do we show (and in an analysis would exclude) the cells that are **cut by the image border**?
2. Do bigger cells have bigger nuclei? Does the N/C ratio vary a lot between cells?
3. We measured only **8 or so complete cells**. Would you draw biological conclusions from this? What would you do in a real experiment?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you have tried</summary>

1. A cell (or nucleus) cut by the border is **only partly measured**, so its area is too small and the ratio is wrong. Both the nucleus **and** the cell should be complete.
2. Usually there is a **positive** trend (bigger cells tend to have bigger nuclei), but with clear spread, so the ratio is not constant. That is why the N/C ratio is useful: it gives a size-normalised view of nuclear size.
3. **No.** A handful of cells is far too few for any statistics. In a real experiment you would analyse **many images and many cells per condition**, use **identical parameters** for all of them, and compare conditions with proper statistics. This notebook is a workflow demo, not a result.

</details>

### Step 4.8 – 🏆 Challenge (optional): how robust is your result?
We saw that the measured area depends on the threshold. **By how much?** The cell below re-runs the nuclear segmentation with thresholds from half to double your chosen value and reports the number of objects and the mean area.

> 💡 This is called a **sensitivity analysis**. If a conclusion changes when you change the threshold slightly, it is not reliable.

In [ ]:
rows = []
for factor in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
    t = factor * nuclei_thresh_used
    _, _, lab, _ = segment_nuclei(nuclei_f, blur_radius, t, min_size)
    props = measure.regionprops(lab)
    inside = [p.area for p in props if not (p.bbox[0] == 0 or p.bbox[1] == 0 or p.bbox[2] == H or p.bbox[3] == W)]
    rows.append({"Threshold factor": factor,
                 "Threshold": round(t, 3),
                 "Objects": int(lab.max()),
                 "Mean area, all (px)": round(np.mean([p.area for p in props]), 1) if props else np.nan,
                 "Mean area, not touching border (px)": round(np.mean(inside), 1) if inside else np.nan})
display(pd.DataFrame(rows))

### 🏆 Question 4.8
1. Which threshold factors give a **stable number of objects**? Which ones do not, and what goes wrong there?
2. How much does the **mean area** change between a factor of 0.75 and 1.25? What does this mean if you want to report "the nuclei are X pixels large"?
3. If you want to compare **two conditions**, why is it essential to use exactly the same parameters for both?

✍️ **Your answer:** *(double-click to write)*

<details>
<summary>✅ Check yourself – open after you looked at the table</summary>

1. Around the chosen value (factors ~0.75–1.5) the object count stays about the same. At **low** factors nuclei start to **merge**, so the count drops and the mean area jumps up; at **high** factors the dim nuclei **disappear**.
2. The mean area changes noticeably, even in the "stable" range, because the outline moves outward or inward when the threshold changes. An **absolute size** is therefore only meaningful together with the **method and parameters** used; a **relative comparison** with identical settings is much more robust.
3. If the parameters differ between conditions, a difference in the result could be produced by the *analysis* instead of the *biology*.

**Extension idea for the curious:** nuclei that touch could be separated with a distance-transform watershed, using the same watershed function you already know.

</details>

---
## 5 · Document and export

### Step 5.1 – Record what you did
Anyone (including *you* in six months) should be able to repeat your analysis. Run the cell to collect the parameters and save your table.

In [ ]:
parameters = pd.Series({
    "Image file":                          url.split("/")[-1],
    "Nuclei: median filter radius (px)":   blur_radius,
    "Nuclei: threshold (0-1 scale)":       round(float(nuclei_thresh_used), 3),
    "Nuclei: minimum object size (px)":    min_size,
    "Cells: mean filter size (px)":        cell_blur,
    "Cells: threshold (0-1 scale)":        round(float(cell_thresh_used), 3),
    "Cells: minimum region size (px)":     cell_min_size,
    "Cell splitting":                      "marker-controlled watershed, markers = nuclei, mask = cell mask",
    "Intensity background (raw units)":    float(background),
    "Pixel size (µm)":                     pixel_size_um if pixel_size_um > 0 else "unknown (values in pixels)",
}, name="value")
display(parameters.to_frame())

df.to_csv("nuclei_measurements.csv", index=False)
parameters.to_csv("analysis_parameters.csv")
try:
    from google.colab import files
    files.download("nuclei_measurements.csv")
    files.download("analysis_parameters.csv")
except ImportError:
    print("Saved nuclei_measurements.csv and analysis_parameters.csv in the current folder.")

### Step 5.2 – Take-home messages

1. **Look at your data first.** Shape, bit depth, intensity range, background and histogram tell you what is possible.
2. **A pipeline is a chain of simple steps:** filter → threshold → clean up → label. Each step has a parameter; each parameter changes the result.
3. **There is no "true" threshold.** Choose one that you can justify by *looking* at the result, apply it consistently, and report it.
4. **Global thresholds struggle with objects of different brightness.** Check what an automatic method really did.
5. **Touching objects need more than a threshold.** Seeds (here: nuclei) + watershed split them.
6. **Segment on a copy, measure on the raw data, and subtract the background** before measuring intensities.
7. **Objects cut by the image border** (and tiny ones) bias size, shape and intensity measurements. Flag and exclude them.
8. **Always do quality control:** overlay the result on the raw image and look for merged, split, missing and truncated objects.
9. **Document your parameters** so that your analysis is reproducible.

### 🚀 Where to go from here
* The same workflow in **Fiji/ImageJ** with the macro-based NEUBIAS tutorial: https://neubias.github.io/training-resources/
* For difficult images (dense, dim, very irregular cells), **deep-learning segmentation** models such as *Cellpose* or *StarDist* often outperform classical pipelines. Pretrained models and easy cloud-based training are available via the **BioImage Model Zoo** (https://bioimage.io/) and **ZeroCostDL4Mic**.
* But note: deep learning needs the **same careful quality control**, and it has its own reproducibility pitfalls (Laine et al., 2021, *Nature Methods*, https://doi.org/10.1038/s41592-021-01284-3).
* Interactive viewing and annotation of your own data: **napari** (https://napari.org/).
* More notebooks: https://haesleinhuepf.github.io/BioImageAnalysisNotebooks/intro.html